## Installs and Imports

In [ ]:
!pip install -U torch torchvision
!pip install transformers datasets tqdm pandas scipy

In [ ]:
!pip install --force-reinstall --no-cache-dir scipy # Only needed within runpod environment
!pip install --force-reinstall --no-cache-dir typing_extensions==4.11.0
!pip uninstall -y Pillow
!pip install Pillow
!pip install numpy==1.26.4

In [ ]:
## Sometimes needed in runpod to make sure it goes to the network volumne
import os

os.environ["HF_DATASETS_CACHE"] = "/workspace/hf_cache"
os.environ["TRANSFORMERS_CACHE"] = "/workspace/hf_cache"
os.environ["HF_HOME"] = "/workspace/hf_home"

In [ ]:
import torchvision
import torch
from torch.utils.data import DataLoader
from tqdm import tqdm
from transformers import CLIPForImageClassification, AutoImageProcessor
from datasets import load_dataset
import numpy as np
import copy
from collections import defaultdict
import pandas as pd
from PIL import Image

## Configuration

In [ ]:
num_class = [47, 10, 43, 10, 45, 196, 397, 10]
dataset_name = ["DTD", "EuroSAT", "GTSRB", "MNIST", "RESISC45", "Stanford_Cars", "SUN397", "SVHN"]
model_name = "CLIP_ViT_Vision"
device = "cuda" if torch.cuda.is_available() else "cpu"
refining_type = "Standard" # "Standard" | "Increment_Training"
refiner = "Fine_Tuned" # "Fine_Tuned" | "Linear_Probe" | "Reverse_Probe"

# https://huggingface.co/openai/clip-vit-base-patch32
refer = CLIPForImageClassification.from_pretrained("openai/clip-vit-base-patch32")
processor = AutoImageProcessor.from_pretrained("openai/clip-vit-base-patch32")

## Datasets

### Torch Vision

#### Downloading

In [ ]:
# DTD - https://docs.pytorch.org/vision/main/generated/torchvision.datasets.DTD.html

train_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    split="train",
    download=True,
    transform=None,
    partition=1,
)

val_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    split="val",
    download=True,
    transform=None,
    partition=1,
)

test_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    split="test",
    download=True,
    transform=None,
    partition=1,
)

In [ ]:
# EuroSAT - https://docs.pytorch.org/vision/main/generated/torchvision.datasets.EuroSAT.html

dataset = torchvision.datasets.EuroSAT(
    root="./Datasets/EuroSAT/raw",
    download=True,
    transform=None,
)

In [ ]:
# GTSRB - https://docs.pytorch.org/vision/main/generated/torchvision.datasets.GTSRB.html

train_dataset = torchvision.datasets.GTSRB(
    root="./Datasets/GTSRB/raw",
    split="train",
    download=True,
    transform=None,
)

test_dataset = torchvision.datasets.GTSRB(
    root="./Datasets/GTSRB/raw",
    split="test",
    download=True,
    transform=None,
)

In [ ]:
# MNIST - https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.MNIST.html

train_dataset = torchvision.datasets.MNIST(
    root="./Datasets/MNIST/raw",
    train=True,
    download=True,
    transform=None,
)

test_dataset = torchvision.datasets.MNIST(
    root="./Datasets/MNIST/raw",
    train=False,
    download=True,
    transform=None,
)

In [ ]:
# SUN397 - https://docs.pytorch.org/vision/main/generated/torchvision.datasets.SUN397.html

dataset = torchvision.datasets.SUN397(
    root="./Datasets/SUN397/raw",
    download=True,
)

In [ ]:
# SVHN - https://docs.pytorch.org/vision/0.19/generated/torchvision.datasets.SVHN.html

train_dataset = torchvision.datasets.SVHN(
    root="./Datasets/SVHN/raw",
    split="train",
    download=True,
)

test_dataset = torchvision.datasets.SVHN(
    root="./Datasets/SVHN/raw",
    split="test",
    download=True,
)

#### Processing

In [ ]:
train_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    version="train",
    download=False,
    transform=None,
    partition=1,
)

val_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    version="val",
    download=False,
    transform=None,
    partition=1,
)

test_dataset = torchvision.datasets.DTD(
    root="./Datasets/DTD/raw",
    version="val",
    download=False,
    transform=None,
    partition=1,
)

### Hugging Face

#### Downloading

In [ ]:
# RESISC45 - https://huggingface.co/datasets/timm/resisc45

dataset = load_dataset("timm/resisc45")

train = dataset["train"]
val = dataset["validation"]
test = dataset["test"]

In [ ]:
# Stanford Cars - https://huggingface.co/datasets/tanganke/stanford_cars

dataset = load_dataset("tanganke/stanford_cars")

split = dataset["train"].train_test_split(test_size=0.2, seed=66)
train = split["train"]
val = split["test"]
test = dataset["test"]